# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mirl0w/Machine-Learning-Intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page, for one client, on one day
(grain: content_hash_id + client_hash_id + report_date).

Time window: developing on a mid-panel month, month=2026-03, to avoid both
cold-start clients (thin early history) and the sealed final month
(June 2026, which must stay untouched as a future test window).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Features (knowable before any decision):
- impressions_90d — observed Search Console count
- clicks — observed Search Console count
- avg_position — observed ranking at time of report
- ctr — derived from impressions/clicks in the SAME window, safe
- sessions — observed GA4 measurement, same report_date

Label/proxy: whether a page's impressions dropped meaningfully within the
observed window (decline indicator) — later upgraded to a future-window
label (prior 90 days -> next 30 days decline).

Context (not used as features, background only):
- client_hash_id, content_hash_id — join keys only, no signal value themselves

Excluded, and why:
- Any FlyRank product decision flags (health_score, priority_score, action_type)
  — not shipped in this data, and I won't reconstruct them, since feeding a
  rebuilt product decision back in as a "feature" would be circular: the model
  would just learn to copy an existing rule instead of finding real signal.
- trend_pct — this is the same trap seen in Notebook 02; it's the label in
  disguise, so it's excluded as a feature (see Section 3, where I deliberately
  break this rule to demonstrate why).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [6]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

# --- Query 1: grain check — one row per content+client+day? ---
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as n
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
""").df()
print("Duplicate grain rows (should be 0):", len(grain_check))

# --- Query 2: row count + date span for this slice ---
span = con.sql(f"""
    SELECT COUNT(*) as row_count, MIN(report_date) as min_date, MAX(report_date) as max_date
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(span)

# --- Query 3: availability check with IS TRUE ---
avail = con.sql(f"""
    SELECT COUNT(*) as rows_with_ga4
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE ga4_data_available IS TRUE
""").df()
print(avail)

# --- Five-feature frame (computing ctr manually, since it's not a stored column) ---
features_query = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        CASE WHEN gsc_impressions > 0
             THEN gsc_clicks * 1.0 / gsc_impressions
             ELSE NULL END AS ctr,
        ga4_sessions
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    LIMIT 1000
""").df()
features_query.head()

# --- The trap: deliberately add a label-derived feature and watch score jump ---
# There's no pre-built trend_pct column in the warehouse (that was starter-CSV only),
# so we build the label AND the leaky feature from the SAME future window on purpose —
# this recreates the same trap: a feature that is the answer in disguise.

leak_demo = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        CASE WHEN gsc_impressions > 0
             THEN gsc_clicks * 1.0 / gsc_impressions
             ELSE NULL END AS ctr,
        ga4_sessions,
        -- LEAKY: this label and this "feature" are both built from gsc_impressions
        -- in the exact same window, so the feature basically restates the label
        CASE WHEN gsc_impressions < 100 THEN 1 ELSE 0 END AS is_declining_label,
        gsc_impressions AS leaky_feature_same_window_impressions
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    LIMIT 5000
""").df()

# Quick "score": correlation between the leaky feature and the label
leak_corr = leak_demo["leaky_feature_same_window_impressions"].corr(leak_demo["is_declining_label"])
print(f"Correlation with leaky feature included: {leak_corr:.3f}  <- suspiciously strong, because it's circular")

# Now remove it and keep only the honest features
honest_features = leak_demo.drop(columns=["leaky_feature_same_window_impressions", "is_declining_label"])
print("\nHonest feature set (leak removed):")
honest_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows (should be 0): 0
   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   rows_with_ga4
0         413966
Correlation with leaky feature included: -0.575  <- suspiciously strong, because it's circular

Honest feature set (leak removed):


,content_hash_id,client_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,ga4_sessions
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,2026-03-01,20,0,3.350000,0.000,<NA>
1,content_05597932fe4da067,client_73cda7b4e4f265ea,2026-03-01,1,0,0.000000,0.000,<NA>
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,2026-03-01,125,1,4.928000,0.008,<NA>
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,2026-03-01,7,0,4.000000,0.000,<NA>
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2026-03-01,11,0,2.272727,0.000,<NA>


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


This slice can never tell me: whether a refresh CAUSES recovery (that needs a
controlled experiment, not observational data). It also can't reflect Google's
actual ranking algorithm — only observed outcomes. The history is an unbalanced
panel: different clients have different amounts of tracked history, and early
rows for some clients are GSC-only (ga4_data_available = FALSE), so GA4-based
features are missing for a portion of rows. My March 2026 slice is also just
one month — seasonal patterns from other months aren't captured here.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.